## Save predictions to the dataframes

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import sklearn
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

In [14]:
SYSTEM_RATED_POWER_DICT = {"HEL" : 21000,
                             "KUO" : 20280,
                             "SOT-20" : 260,
                             "SOT-90" : 260,
                             "TKU" : 4500}

In [15]:
def find_project_root(start: Path = None) -> Path:
    """
    Walk upwards until we find a folder that contains 'data'.
    """
    start = start or Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    raise FileNotFoundError("Could not find project root containing 'data'.")

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models" / "hgbr"

ROOT, DATA_DIR, MODELS_DIR

(WindowsPath('c:/Users/viantt/git2/EUPVSEC-26'),
 WindowsPath('c:/Users/viantt/git2/EUPVSEC-26/data'),
 WindowsPath('c:/Users/viantt/git2/EUPVSEC-26/models/hgbr'))

In [16]:
def pick_city_file(city_tag: str, folder: str = "filtered") -> Path:
    folder_path = DATA_DIR / folder
    matches = sorted(folder_path.glob(f"*{city_tag}*.csv"))
    if not matches:
        raise FileNotFoundError(f"No CSV found for city tag '{city_tag}' in {folder_path}")
    return matches[0]


from pathlib import Path

def pick_model(city_code: str, folder: str, model_type: str = "hgbr") -> Path:
    """
    city_code: HEL, KUO, SOT-20_5, SOT-90_5, TKU, etc.
    folder: "filtered" or "unfiltered"
    model_type: 'hgbr' or 'mlp'
    """
    pattern = f"{model_type}_pipeline_{city_code}.pkl"

    folder_dir = Path(MODELS_DIR) / folder  # robust Path joining
    candidates = sorted(folder_dir.glob(pattern))

    if not candidates:
        raise FileNotFoundError(f"No model files matching '{pattern}' in {folder_dir}")

    # Return most recently modified model
    return max(candidates, key=lambda p: p.stat().st_mtime)


def get_expected_features(pipeline):
    """
    Try a few common sklearn attributes to retrieve expected feature names.
    """
    # Case 1: pipeline saved with feature_names_in_
    if hasattr(pipeline, "feature_names_in_"):
        return list(pipeline.feature_names_in_)
    
    # Case 2: pipeline is a Pipeline and first step might have feature_names_in_
    if hasattr(pipeline, "named_steps"):
        for step in pipeline.named_steps.values():
            if hasattr(step, "feature_names_in_"):
                return list(step.feature_names_in_)
    
    # If we can't infer, return None and we'll fall back to numeric columns
    return None


def build_X(df: pd.DataFrame, pipeline):
    expected = get_expected_features(pipeline)
    if expected is not None:
        missing = [c for c in expected if c not in df.columns]

        # If system doesn't measure POA, POA comp(uted) needs to be used
        if ("poa_comp_rc" in df.columns) & ("poa_rc" in missing):
            df["poa_rc"] = df["poa_comp_rc"]
            missing.remove("poa_rc")

        if missing:
            raise ValueError(
                "Your data is missing columns expected by the model:\n"
                + "\n".join(missing[:50]) + ("" if len(missing) <= 50 else f"\n... (+{len(missing)-50} more)")
            )

        X = df[expected].copy()
        
        return X
    
    # Fallback: use numeric columns except obvious targets
    drop_candidates = {"y", "target", "pv", "PV", "power", "Power", "label"}
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [c for c in numeric_cols if c not in drop_candidates]
    if len(numeric_cols) == 0:
        raise ValueError("Could not infer features; no numeric columns found.")
    return df[numeric_cols].copy()


In [17]:
CITY_MAP = {
    "Helsinki": ("HEL", "FMI_Helsinki"),
    "Kuopio": ("KUO", "FMI_Kuopio"),
    "Turku": ("TKU", "TUAS_Turku"),
    "Sodankyla_20deg": ("SOT-20", "FMI_Sodankyla_20deg"),
    "Sodankyla_90deg": ("SOT-90", "FMI_Sodankyla_90deg"),
}

def run_city(city_name: str, folder="filtered", model_type_list=None, save_df_path=None):
    city_code, file_tag = CITY_MAP[city_name]
    
    # Pick data file
    search_dir = DATA_DIR / "original" / folder
    matches = sorted(search_dir.glob(f"*{file_tag}*.csv"))

    if not matches:
        raise FileNotFoundError(f"No data file for {city_name} in {search_dir}")
    
    data_path = matches[0]
    df = pd.read_csv(data_path, sep=";")

    df_out = df.copy()
    df_out = df_out.set_index("utctime")
    df_out.index = pd.to_datetime(df_out.index, errors="coerce")
    
    # Pick model
    for model_type in model_type_list:
        # model_gen_path = pick_model(city_code=city_code, folder=folder, model_type=model_type)
        # model_gen = joblib.load(model_gen_path)
        
        model_fit_path = pick_model(city_code=city_code, folder=folder, model_type=model_type)
        model_fit = joblib.load(model_fit_path)

        # Predict
        # X_gen = build_X(df, model_gen)
        # pred_gen = model_gen.predict(X_gen)

        X_fit = build_X(df, model_fit)
        pred_fit = model_fit.predict(X_fit)

       # df_out[f"{model_type}_gen"] = pred_gen * SYSTEM_RATED_POWER_DICT[city_code]
        df_out[f"{model_type}_fit"] = pred_fit * SYSTEM_RATED_POWER_DICT[city_code]

    
    return df_out

In [18]:
# --- Helsinki ---

df_hel_filtered = run_city(
    city_name="Helsinki", 
    folder="filtered", 
    model_type_list=["hgbr"],
    save_df_path="data/with_predictions"
    )

df_hel_unfiltered = run_city(
    city_name="Helsinki", 
    folder="unfiltered", 
    model_type_list=["hgbr"],
    save_df_path="data/with_predictions"
    )

In [19]:
# --- Kuopio ---
df_kuo_filtered = run_city(
    city_name="Kuopio", 
    folder="filtered", 
    model_type_list=["hgbr"],
    save_df_path="data/with_predictions"
    )

df_kuo_unfiltered= run_city(
    city_name="Kuopio", 
    folder="unfiltered", 
    model_type_list=["hgbr"],
    save_df_path="data/with_predictions"
    )

In [20]:
# --- Sodankyla_20deg ---
df_sod20_filtered = run_city(
    city_name="Sodankyla_20deg", 
    folder="filtered", 
    model_type_list=["hgbr"],
    save_df_path="data/with_predictions"
    )

df_sod20_unfiltered = run_city(
    city_name="Sodankyla_20deg", 
    folder="unfiltered", 
    model_type_list=["hgbr"],
    save_df_path="data/with_predictions"
    )

In [21]:
# --- Sodankyla_90deg ---
df_sod90_filtered = run_city(
    city_name="Sodankyla_90deg", 
    folder="filtered", 
    model_type_list=["hgbr"],
    save_df_path="data/with_predictions"
    )

df_sod90_unfiltered = run_city(
    city_name="Sodankyla_90deg", 
    folder="unfiltered", 
    model_type_list=["hgbr"],
    save_df_path="data/with_predictions"
    )

In [22]:
# --- Turku ---
df_tku_filtered = run_city(
    city_name="Turku", 
    folder="filtered", 
    model_type_list=["hgbr"],
    save_df_path="data/with_predictions"
    )

df_tku_unfiltered = run_city(
    city_name="Turku", 
    folder="unfiltered", 
    model_type_list=["hgbr"],
    save_df_path="data/with_predictions"
    )

## Calculate Huld

In [23]:
import datetime
from scipy.optimize import curve_fit
from functools import partial


In [24]:
def estimate_huld(rated_power, data, radiation_col_name="poa_rc", power_col_name="power", training_year=0):
    """
    Huld regression model is a predictive model using module temperature and in-plane irradiance [1].
    Authors fit the model to indoor data but here outdoor data is used. Coefficients are obtained from
    one training year, after which performance losses can be tracked by comparing the measured powers to
    predicted ones.
    Args:
        data (Pandas DataFrame): input dataframe
        training_year (int): The number of the year the data is trained. If 0, default coefficients k1-k6 will be used.
    Returns:
 
    References:
        [1] Thomas Huld et al. A power-rating model for crystalline silicon PV modules,
        Solar Energy Materials and Solar Cells, Volume 95, Issue 12, 2011, Pages 3359-3369, ISSN 0927-0248,
        https://doi.org/10.1016/j.solmat.2011.07.026.
    """
 
    # data = data.dropna()  # Drop nan values rowwise to be able to perform the fitting
 
    # If module temperature column not in data, use cell temperature
    if 'module_temp' not in data.columns:
        data['module_temp'] = data['cell_temp'].copy()

    # huld 2010 constants
    k1 = -0.017162
    k2 = -0.040289
    k3 = -0.004681
    k4 = 0.000148
    k5 = 0.000169
    k6 = 0.000005

    default_coeffs = [k1, k2, k3, k4, k5, k6]

    # If training years are given, fit the coefficients. Otherwise, use the default coefficients for predictions.
    if training_year > 0:
        training_data = data[data.index <= data.index[0] + datetime.timedelta(days=365*training_year)]
        testing_data = data[data.index > data.index[0] + datetime.timedelta(days=365*training_year)]

        print(f"Training data len: {training_data.shape[0]}")

        i, t, p = (training_data[radiation_col_name]).values.astype('float64'), \
                (training_data['module_temp']).values.astype('float64'), \
                (training_data[power_col_name]).values.astype('float64')
        # Fit to data and find the coefficients
        #coeffs, pcov = curve_fit(partial(_huld, config.rated_power*1000), [i, t], p)
        #coeffs, pcov = curve_fit(f=_huld,
        #                         xdata=[config.rated_power*1000, [i, t]],
        #                         ydata=p,
        #                         method="lm")
        coeffs, pcov = curve_fit(f=partial(_huld, rated_power), 
                                 xdata=[i, t], 
                                 ydata=p,
                                 p0=default_coeffs,
                                 method="lm")

        print(f"Huld fitted coefficients (scaled): {np.array(coeffs) / rated_power}")
    else:
        coeffs = default_coeffs
        testing_data = data
        
    # Make the predictions for the whole dataset (including the years used for fitting)
    pred_p = _huld(rated_power, [testing_data[radiation_col_name], testing_data['module_temp']],
                   coeffs[0], coeffs[1], coeffs[2], coeffs[3], coeffs[4], coeffs[5])
   
    # # Calculate the performance points by dividing output power with the predicted power
    # norm_p = data.power / pred_p.values
    # 
    # return G_weighted_aggregation(interval, data['poa'], norm_p)
    return pred_p


def _huld(rated_power, X, k1, k2, k3, k4, k5, k6):
    """
    Args:
        p_rated (float): nameplate power rating of the PV system in watts
        X:
        k1-k6:
    Returns:
    """
    ref_temp = 25
    ref_irrad = 1000
 
    i, t = X
    G = i / ref_irrad
    G[G <= 0] = 0.000001 # Make zeros small floats for log
    T = t - ref_temp
    pred_p = G*(rated_power + k1*np.log(G) + k2*np.log(G)**2 + k3*T + k4*T*np.log(G) + k5*T*np.log(G)**2 + k6*T**2)
   
    return pred_p

In [25]:
def estimate_empirical_models(data, rated_power, radiation_col_name, power_col_name, training_year=1):
    data["Huld_gen"] = estimate_huld(rated_power, data, radiation_col_name=radiation_col_name, power_col_name=power_col_name)
    data["Huld_fit"] = estimate_huld(rated_power, data, radiation_col_name=radiation_col_name, power_col_name=power_col_name, training_year=training_year)
    
    return data

In [26]:
df_hel_filtered = estimate_empirical_models(df_hel_filtered, SYSTEM_RATED_POWER_DICT["HEL"],
                                            radiation_col_name="poa_rc",
                                            power_col_name="power",
                                            training_year=1)

df_hel_unfiltered = estimate_empirical_models(df_hel_unfiltered, SYSTEM_RATED_POWER_DICT["HEL"],
                                              radiation_col_name="poa_rc",
                                              power_col_name="power",
                                              training_year=1)


Training data len: 177993
Huld fitted coefficients (scaled): [ 1.30773060e-01  5.94396367e-02 -6.47648075e-03 -5.19128569e-03
  7.08071294e-04 -2.94103482e-06]
Training data len: 219165
Huld fitted coefficients (scaled): [ 0.14565287  0.06936739  0.00677786  0.0022593   0.00029416 -0.00072226]


In [27]:
df_kuo_filtered = estimate_empirical_models(df_kuo_filtered, SYSTEM_RATED_POWER_DICT["KUO"],
                                            radiation_col_name="poa_rc",
                                            power_col_name="power",
                                            training_year=1)

df_kuo_unfiltered = estimate_empirical_models(df_kuo_unfiltered, SYSTEM_RATED_POWER_DICT["KUO"],
                                              radiation_col_name="poa_rc",
                                              power_col_name="power",
                                              training_year=1)

Training data len: 125409
Huld fitted coefficients (scaled): [ 1.22262143e-01  4.13856237e-02 -5.25629384e-03  1.28218394e-04
  1.56028344e-03  5.32520043e-05]
Training data len: 149096
Huld fitted coefficients (scaled): [ 0.1332563   0.06370062  0.00092812  0.00085104  0.00170077 -0.00031211]


In [28]:
df_sod20_filtered = estimate_empirical_models(df_sod20_filtered, SYSTEM_RATED_POWER_DICT["SOT-20"],
                                            radiation_col_name="poa_comp_rc",
                                            power_col_name="power",
                                            training_year=1)

df_sod20_unfiltered = estimate_empirical_models(df_sod20_unfiltered, SYSTEM_RATED_POWER_DICT["SOT-20"],
                                              radiation_col_name="poa_comp_rc",
                                              power_col_name="power",
                                              training_year=1)

Training data len: 162075
Huld fitted coefficients (scaled): [ 3.81912965e-01  1.70256060e-01 -1.72274594e-03  1.21166088e-02
  8.25758313e-03 -2.11111146e-04]
Training data len: 200357
Huld fitted coefficients (scaled): [ 0.38678286  0.2060039   0.00231557  0.01579152  0.00982098 -0.00039477]


In [29]:
df_sod90_filtered = estimate_empirical_models(df_sod90_filtered, SYSTEM_RATED_POWER_DICT["SOT-90"],
                                            radiation_col_name="poa_rc",
                                            power_col_name="power",
                                            training_year=1)

df_sod90_unfiltered = estimate_empirical_models(df_sod90_unfiltered, SYSTEM_RATED_POWER_DICT["SOT-90"],
                                              radiation_col_name="poa_rc",
                                              power_col_name="power",
                                              training_year=1)

Training data len: 183376
Huld fitted coefficients (scaled): [ 2.09180519e-01  5.08336103e-02 -2.39501030e-03  5.80686350e-03
  2.96620410e-03 -1.84967832e-04]
Training data len: 198309
Huld fitted coefficients (scaled): [ 2.06837222e-01  4.74745459e-02 -2.49291936e-03  5.52727141e-03
  2.81009572e-03 -1.90944468e-04]


In [30]:
df_tku_filtered = estimate_empirical_models(data=df_tku_filtered, 
                                            rated_power=SYSTEM_RATED_POWER_DICT["TKU"],
                                            radiation_col_name="poa_comp_rc",
                                            power_col_name="power",
                                            training_year=1)

df_tku_unfiltered = estimate_empirical_models(df_tku_unfiltered, 
                                              SYSTEM_RATED_POWER_DICT["TKU"],
                                              radiation_col_name="poa_comp_rc",
                                              power_col_name="power",
                                              training_year=1)

Training data len: 35454
Huld fitted coefficients (scaled): [ 1.87763766e-01  5.89090454e-02 -9.21302433e-03 -9.87065390e-03
 -2.38597962e-04  3.05549915e-05]
Training data len: 40165
Huld fitted coefficients (scaled): [ 0.30824446  0.13645629  0.00296025 -0.00722331 -0.00110768 -0.00052384]


## Save dataframes

In [31]:
def save_df(df, save_df_path=None):
    if save_df_path:
        #save_df_path = Path(save_df_path)
        #save_df_path.mkdir(parents=True, exist_ok=True)

        print(f"Saving df to path: {save_df_path}")
        df.to_csv(save_df_path, sep=";")

In [32]:
save_df(df_hel_filtered, "data/with_predictions/filtered/FMI_Helsinki_PV_filtered_daytime-iec-threshold0_20000-qc-snow-outlier800_2_with_preds.csv")
save_df(df_hel_unfiltered, "data/with_predictions/unfiltered/FMI_Helsinki_PV_filtered_daytime_with_preds.csv")

Saving df to path: data/with_predictions/filtered/FMI_Helsinki_PV_filtered_daytime-iec-threshold0_20000-qc-snow-outlier800_2_with_preds.csv
Saving df to path: data/with_predictions/unfiltered/FMI_Helsinki_PV_filtered_daytime_with_preds.csv


In [33]:
save_df(df_kuo_filtered, "data/with_predictions/filtered/FMI_Kuopio_PV_filtered_daytime-iec-threshold0_20000-qc-snow-outlier800_2_with_preds.csv")
save_df(df_kuo_unfiltered, "data/with_predictions/unfiltered/FMI_Kuopio_PV_filtered_daytime_with_preds.csv")

Saving df to path: data/with_predictions/filtered/FMI_Kuopio_PV_filtered_daytime-iec-threshold0_20000-qc-snow-outlier800_2_with_preds.csv
Saving df to path: data/with_predictions/unfiltered/FMI_Kuopio_PV_filtered_daytime_with_preds.csv


In [34]:
save_df(df_sod20_filtered, "data/with_predictions/filtered/FMI_Sodankyla_20deg_PV_filtered_daytime-iec-threshold0_20000-qc-snow-outlier800_2-cutoulier400_800_800_with_preds.csv")
save_df(df_sod20_unfiltered, "data/with_predictions/unfiltered/FMI_Sodankyla_20deg_PV_filtered_daytime_with_preds.csv")

Saving df to path: data/with_predictions/filtered/FMI_Sodankyla_20deg_PV_filtered_daytime-iec-threshold0_20000-qc-snow-outlier800_2-cutoulier400_800_800_with_preds.csv
Saving df to path: data/with_predictions/unfiltered/FMI_Sodankyla_20deg_PV_filtered_daytime_with_preds.csv


In [35]:
save_df(df_sod90_filtered, "data/with_predictions/filtered/FMI_Sodankyla_90deg_PV_filtered_daytime-iec-threshold0_20000-qc-snow-outlier800_2_with_preds.csv")
save_df(df_sod90_unfiltered, "data/with_predictions/unfiltered/FMI_Sodankyla_90deg_PV_filtered_daytime_with_preds.csv")

Saving df to path: data/with_predictions/filtered/FMI_Sodankyla_90deg_PV_filtered_daytime-iec-threshold0_20000-qc-snow-outlier800_2_with_preds.csv
Saving df to path: data/with_predictions/unfiltered/FMI_Sodankyla_90deg_PV_filtered_daytime_with_preds.csv


In [36]:
save_df(df_tku_filtered, "data/with_predictions/filtered/TUAS_Turku_PV_filtered_daytime-iec-threshold0100_1002000-qc-snow-outlier800_2_with_preds.csv")
save_df(df_tku_unfiltered, "data/with_predictions/unfiltered/TUAS_Turku_PV_filtered_daytime_with_preds.csv")

Saving df to path: data/with_predictions/filtered/TUAS_Turku_PV_filtered_daytime-iec-threshold0100_1002000-qc-snow-outlier800_2_with_preds.csv
Saving df to path: data/with_predictions/unfiltered/TUAS_Turku_PV_filtered_daytime_with_preds.csv
